In [18]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
np.random.seed(42)


In [19]:
DATA_DIR = Path("../output")
ARTIFACT_DIR = Path("../artifacts")

orders = pd.read_csv(DATA_DIR / "orders.csv")
inventory = pd.read_csv(DATA_DIR / "inventory_events.csv")
vendors = pd.read_csv(DATA_DIR / "vendors.csv")
forecast_df = pd.read_csv(ARTIFACT_DIR / "weekly_forecast_future.csv")

orders.head()



,order_id,order_date,sku_id,region,payment_type,delivery_days,delivery_status,campaign_applied,order_quantity
0,ORD000000104,0,SKU0002,south,COD,4,DELIVERED,False,1
1,ORD000000103,0,SKU0002,south,COD,3,DELIVERED,False,1
2,ORD000000102,0,SKU0002,south,COD,2,DELIVERED,False,1
3,ORD000000101,0,SKU0002,south,COD,1,DELIVERED,False,1
4,ORD000000100,0,SKU0002,north,PREPAID,5,DELIVERED,False,1


In [20]:
inventory_snapshot = (
    inventory
    .groupby(["sku_id", "warehouse"])
    .agg(current_stock=("quantity", "sum"))
    .reset_index()
)

inventory_snapshot.rename(columns={"warehouse": "warehouse_id"}, inplace=True)
inventory_snapshot


,sku_id,warehouse_id,current_stock
0,SKU0001,WH_east,38
1,SKU0001,WH_north,0
2,SKU0001,WH_south,25
3,SKU0001,WH_west,37
4,SKU0002,WH_east,1
5,SKU0002,WH_north,22
6,SKU0002,WH_south,36
7,SKU0002,WH_west,49
8,SKU0003,WH_east,36
9,SKU0003,WH_north,183


In [21]:
vendors = vendors.copy()

assert "lead_time_days" in vendors.columns, "vendors.csv missing lead_time_days"

vendors["lead_time_weeks"] = (
    vendors["lead_time_days"] / 7
).apply(np.ceil).astype(int)

vendors[["sku_id", "lead_time_days", "lead_time_weeks", "MOQ"]]


,sku_id,lead_time_days,lead_time_weeks,MOQ
0,SKU0001,13,2,50
1,SKU0002,14,2,50
2,SKU0003,25,4,200
3,SKU0004,30,5,50
4,SKU0005,30,5,200


In [22]:
sku_history = (
    orders
    .groupby("sku_id")["order_date"]
    .nunique()
    .div(7)
    .astype(int)
    .rename("history_weeks")
    .reset_index()
)

sku_history


,sku_id,history_weeks
0,SKU0001,104
1,SKU0002,104
2,SKU0003,104
3,SKU0004,104
4,SKU0005,104


In [23]:
def route_sku(history_weeks: int) -> str:
    if history_weeks >= 60:
        return "SARIMAX"
    elif history_weeks >= 12:
        return "ML_BASELINE"
    else:
        return "NAIVE"


In [24]:
sku_map = sku_history.copy()
sku_map["forecast_strategy"] = sku_map["history_weeks"].apply(route_sku)

sku_map["sku_status"] = sku_map["forecast_strategy"].map({
    "SARIMAX": "READY",
    "ML_BASELINE": "LIMITED",
    "NAIVE": "NEW"
})

sku_map


,sku_id,history_weeks,forecast_strategy,sku_status
0,SKU0001,104,SARIMAX,READY
1,SKU0002,104,SARIMAX,READY
2,SKU0003,104,SARIMAX,READY
3,SKU0004,104,SARIMAX,READY
4,SKU0005,104,SARIMAX,READY


In [25]:
context = (
    inventory_snapshot
    .merge(vendors, on="sku_id", how="left")
    .merge(sku_map, on="sku_id", how="left")
)

# Final inventory position
context["inventory_position"] = context["current_stock"]

# Hard schema guardrails
required_cols = {
    "sku_id",
    "warehouse_id",
    "lead_time_weeks",
    "MOQ",
    "forecast_strategy",
    "inventory_position"
}
missing = required_cols - set(context.columns)
assert not missing, f"Context missing columns: {missing}"

context


,sku_id,warehouse_id,current_stock,vendor_id,lead_time_days,MOQ,unit_cost,lead_time_weeks,history_weeks,forecast_strategy,sku_status,inventory_position
0,SKU0001,WH_east,38,V_KU0001,13,50,224.26,2,104,SARIMAX,READY,38
1,SKU0001,WH_north,0,V_KU0001,13,50,224.26,2,104,SARIMAX,READY,0
2,SKU0001,WH_south,25,V_KU0001,13,50,224.26,2,104,SARIMAX,READY,25
3,SKU0001,WH_west,37,V_KU0001,13,50,224.26,2,104,SARIMAX,READY,37
4,SKU0002,WH_east,1,V_KU0002,14,50,617.01,2,104,SARIMAX,READY,1
5,SKU0002,WH_north,22,V_KU0002,14,50,617.01,2,104,SARIMAX,READY,22
6,SKU0002,WH_south,36,V_KU0002,14,50,617.01,2,104,SARIMAX,READY,36
7,SKU0002,WH_west,49,V_KU0002,14,50,617.01,2,104,SARIMAX,READY,49
8,SKU0003,WH_east,36,V_KU0003,25,200,105.18,4,104,SARIMAX,READY,36
9,SKU0003,WH_north,183,V_KU0003,25,200,105.18,4,104,SARIMAX,READY,183


In [ ]:
def demand_from_forecast(forecast_df, sku_id, lead_time_weeks):
    # forecast_df contains SKU-level p50/p90 forecasts per week; sum across lead time
    df = forecast_df[forecast_df["sku_id"] == sku_id].head(lead_time_weeks)
    assert len(df) > 0, f"No forecast found for {sku_id}"
    expected = df["p50"].sum()
    safety = df["p90"].sum() - df["p50"].sum()
    # Return SKU-level totals with explicit demand_source label
    return {
        "expected_demand_LT": expected,
        "safety_stock": safety,
        "demand_source": "PRIMARY_SARIMAX"
    }


def demand_from_history(orders, sku_id, lead_time_weeks):
    # Simple history fallback: weekly aggregation from orders
    weekly = (
        orders[orders["sku_id"] == sku_id]
        .groupby(orders["order_date"] // 7)["order_quantity"]
        .sum()
    )
    avg = weekly.mean() if len(weekly) > 0 else 0.0
    return {
        "expected_demand_LT": float(avg) * lead_time_weeks,
        "safety_stock": float(avg) * 0.3 * lead_time_weeks,
        "demand_source": "FALLBACK_HISTORY"
    }


def demand_for_new_sku(lead_time_weeks):
    # Naive fallback for new SKUs
    BASE = 100
    return {
        "expected_demand_LT": float(BASE) * lead_time_weeks,
        "safety_stock": float(BASE) * 0.5 * lead_time_weeks,
        "demand_source": "FALLBACK_NAIVE"
    }


In [ ]:
def resolve_demand(row):
    # Ensure lead time available
    assert not pd.isna(row["lead_time_weeks"]), f"Missing lead_time_weeks for {row['sku_id']}"

    if row["forecast_strategy"] == "SARIMAX":
        return demand_from_forecast(
            forecast_df,
            row["sku_id"],
            row["lead_time_weeks"]
        )
    elif row["forecast_strategy"] == "ML_BASELINE":
        return demand_from_history(
            orders,
            row["sku_id"],
            row["lead_time_weeks"]
        )
    else:
        return demand_for_new_sku(row["lead_time_weeks"])


In [ ]:
# Compute warehouse shares per SKU so we allocate SKU-level demand across warehouses.
# We prefer using historical outbound orders per (sku, warehouse) where available;
# otherwise fall back to current inventory distribution as a proxy; if neither
# is available, split demand evenly across warehouses. This prevents demand
# multiplication when applying SKU demand to every warehouse independently.

# Helper: compute per-sku warehouse shares
warehouse_shares = {}
# Detect warehouse column in orders if present
orders_warehouse_col = next((c for c in orders.columns if 'warehouse' in c.lower()), None)
if orders_warehouse_col is not None:
    # Use historical shipped quantities to compute share
    grp = (
        orders
        .groupby(['sku_id', orders_warehouse_col])['order_quantity']
        .sum()
        .reset_index(name='qty')
    )
    for sku, g in grp.groupby('sku_id'):
        total = g['qty'].sum()
        if total <= 0:
            # will compute later from inventory snapshot
            continue
        shares = {row[orders_warehouse_col]: float(row['qty']) / float(total) for _, row in g.iterrows()}
        warehouse_shares[sku] = shares

# Fallback: use inventory_snapshot current_stock distribution
for sku, g in inventory_snapshot.groupby('sku_id'):
    if sku in warehouse_shares:
        continue
    g2 = g.copy()
    g2['qty'] = g2['current_stock'].fillna(0).astype(float)
    total = g2['qty'].sum()
    if total > 0:
        shares = {row['warehouse_id']: float(row['qty']) / float(total) for _, row in g2.iterrows()}
    else:
        # equal split among warehouses for this SKU
        ids = g2['warehouse_id'].tolist()
        if len(ids) == 0:
            shares = {}
        else:
            eq = 1.0 / len(ids)
            shares = {wid: eq for wid in ids}
    warehouse_shares[sku] = shares

# Final safety: ensure every SKU in context has a share map
for sku in context['sku_id'].unique():
    if sku not in warehouse_shares:
        # find warehouses for this sku in context
        wids = context[context['sku_id'] == sku]['warehouse_id'].unique().tolist()
        if not wids:
            warehouse_shares[sku] = {}
        else:
            eq = 1.0 / len(wids)
            warehouse_shares[sku] = {wid: eq for wid in wids}

# Now build decisions per (sku, warehouse) using allocated demand
results = []
errors = []

for idx, row in context.iterrows():
    sku = row['sku_id']
    wid = row['warehouse_id']
    # Resolve SKU-level demand totals (may raise/assert) and preserve source label
    try:
        sku_demand = resolve_demand(row)
    except AssertionError as ae:
        # Missing lead time or forecast: try history fallback then naive
        try:
            sku_demand = demand_from_history(orders, sku, row['lead_time_weeks'])
            sku_demand['demand_source'] = 'FALLBACK_HISTORY'
        except Exception as he:
            sku_demand = demand_for_new_sku(row['lead_time_weeks'])
            sku_demand['demand_source'] = 'FALLBACK_NAIVE'
            errors.append({'sku_id': sku, 'warehouse_id': wid, 'error': str(ae), 'fallback_error': str(he)})
    except Exception as e:
        errors.append({'sku_id': sku, 'warehouse_id': wid, 'error': str(e)})
        sku_demand = demand_for_new_sku(row['lead_time_weeks'])
        sku_demand['demand_source'] = 'ERROR_FALLBACK'

    # Get warehouse share for this SKU; default to equal share across context warehouses
    shares = warehouse_shares.get(sku, {})
    share = shares.get(wid)
    if share is None:
        # equal split among warehouses appearing in context for this sku
        wids = context[context['sku_id'] == sku]['warehouse_id'].unique().tolist()
        if len(wids) == 0:
            share = 1.0
        else:
            share = 1.0 / len(wids)

    # Allocate SKU totals to warehouse by share
    expected_total = float(sku_demand.get('expected_demand_LT', 0.0))
    safety_total = float(sku_demand.get('safety_stock', 0.0))
    allocated_expected = expected_total * share
    allocated_safety = safety_total * share

    # Sanity: ensure allocations sum to SKU totals if we later aggregate (floating-point rounding tolerated)
    # Compute reorder logic using warehouse-level allocated demand
    reorder_point = allocated_expected + allocated_safety
    reorder_required = float(row.get('inventory_position', 0.0)) <= reorder_point

    gap = reorder_point - float(row.get('inventory_position', 0.0))
    if reorder_required and row.get('MOQ', 0) > 0:
        recommended_qty = max(row['MOQ'], int(np.ceil(gap / row['MOQ']) * row['MOQ']))
    else:
        recommended_qty = 0

    results.append({
        'sku_id': sku,
        'warehouse_id': wid,
        'demand_source': sku_demand.get('demand_source', 'UNKNOWN'),
        'inventory_position': float(row.get('inventory_position', 0.0)),
        # warehouse-level allocated values
        'expected_demand_LT': round(allocated_expected, 2),
        'safety_stock': round(allocated_safety, 2),
        'reorder_point': round(reorder_point, 2),
        'reorder_required': bool(reorder_required),
        'recommended_order_qty': int(recommended_qty),
        'sku_status': row.get('sku_status')
    })

decision_df = pd.DataFrame(results)

# Observability: show first few fallbacks/errors
if errors:
    print('Warnings / fallbacks during demand resolution (sample up to 10):')
    for e in errors[:10]:
        print(e)

# Verification checks to run in the notebook (do not raise in production)
# 1) total SKU demand equals sum of allocated warehouse demand (within float tolerance)
sku_agg = decision_df.groupby('sku_id')['expected_demand_LT'].sum().rename('allocated_sum')
# Compare to forecast_df / history totals for a quick sanity check when available
# We'll compute a small diagnostics frame
try:
    # build SKU-level totals from forecast_df where possible
    sku_totals = forecast_df.groupby('sku_id')['p50'].apply(lambda s: s.head(FORECAST_HORIZON).sum()).rename('forecast_p50_sum')
    diag = pd.concat([sku_totals, sku_agg], axis=1).fillna(0.0)
    diag['diff'] = diag['forecast_p50_sum'] - diag['allocated_sum']
    if (diag['diff'].abs() > 1e-6).any():
        print('\nSanity check: some SKU allocated sums differ from forecast totals (may be due to fallbacks).')
        print(diag.head(10))
except Exception:
    # ignore if forecast_df not containing expected rows
    pass

# Final output ready for JSON serialization
decision_df


,sku_id,warehouse_id,strategy_used,inventory_position,expected_demand_LT,safety_stock,reorder_point,reorder_required,recommended_order_qty,sku_status
0,SKU0001,WH_east,SARIMAX,38,895.91,59.70,955.61,True,950,READY
1,SKU0001,WH_north,SARIMAX,0,895.91,59.70,955.61,True,1000,READY
2,SKU0001,WH_south,SARIMAX,25,895.91,59.70,955.61,True,950,READY
3,SKU0001,WH_west,SARIMAX,37,895.91,59.70,955.61,True,950,READY
4,SKU0002,WH_east,FALLBACK_HISTORY,1,928.21,278.46,1206.67,True,1250,READY
5,SKU0002,WH_north,FALLBACK_HISTORY,22,928.21,278.46,1206.67,True,1200,READY
6,SKU0002,WH_south,FALLBACK_HISTORY,36,928.21,278.46,1206.67,True,1200,READY
7,SKU0002,WH_west,FALLBACK_HISTORY,49,928.21,278.46,1206.67,True,1200,READY
8,SKU0003,WH_east,FALLBACK_HISTORY,36,1862.78,558.83,2421.62,True,2400,READY
9,SKU0003,WH_north,FALLBACK_HISTORY,183,1862.78,558.83,2421.62,True,2400,READY
